# OSTutorLLM — Colab API for Fine-tuned Qwen3-8B

This notebook loads the adapter saved by your existing fine-tuning notebook and exposes a prompt API plus optional PDF upload/retrieval. Run cells in order. A Colab GPU runtime is recommended.

**Prerequisites:** Your fine-tuning notebook must have saved the LoRA adapter to `MyDrive/OSTutorLLM/ostutor_qwen3_8b_300_final`. This API notebook does not retrain the model.

**Demo security:** The ngrok URL is public while active. Do not put secrets in prompts or upload confidential documents. Uploaded PDF text stays in memory until the runtime restarts or the document is cleared.


In [ ]:
# OSTutorLLM API — load dependencies
!pip -q install fastapi uvicorn pyngrok pypdf scikit-learn python-multipart peft bitsandbytes transformers


In [ ]:
# Mount Google Drive and configure the saved LoRA adapter path
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/OSTutorLLM"
ADAPTER_PATH = os.path.join(PROJECT_DIR, "ostutor_qwen3_8b_300_final")
BASE_MODEL = "Qwen/Qwen3-8B"

assert os.path.isdir(ADAPTER_PATH), (
    f"Adapter folder not found: {ADAPTER_PATH}. "
    "Run the fine-tuning notebook and save the adapter first."
)
print("Adapter folder:", ADAPTER_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter folder: /content/drive/MyDrive/OSTutorLLM/ostutor_qwen3_8b_300_final


In [ ]:
# Load the base model in 4-bit mode and attach your saved LoRA adapter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer_path = ADAPTER_PATH
try:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("Base model loaded and LoRA adapter attached.")
print("Device:", model.device)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Base model loaded and LoRA adapter attached.
Device: cuda:0


In [ ]:

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import torch
import re

# Create FastAPI application
app = FastAPI(
    title="OSTutorLLM API",
    version="1.0"
)

# Enable requests from your Vercel frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Content-Type"],
)


class PromptRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=4000)


class PromptResponse(BaseModel):
    answer: str


@app.get("/")
def home():
    return {
        "status": "ok",
        "service": "OSTutorLLM",
        "model": "Fine-tuned Qwen3-8B"
    }


@app.post("/generate", response_model=PromptResponse)
def generate_answer(request: PromptRequest):

    question = request.prompt.strip()

    if not question:
        raise HTTPException(
            status_code=400,
            detail="Please enter an Operating Systems question."
        )

    # Give the model clear instructions
    messages = [
        {
            "role": "system",
            "content": (
                "You are OSTutorLLM, an accurate Operating Systems tutor. "
                "Answer every part of the student's question. "
                "Use clear explanations suitable for a university student. "
                "For numerical problems, show the formula, calculate "
                "each requested value, verify the arithmetic, and give "
                "the final answer with units. "
                "Follow the exact order and values stated in the question. "
                "Do not invent different process orders or values. "
                "Do not stop after an introduction. "
                "Keep the answer focused and complete."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    try:
        # Format the conversation for Qwen3
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        # Tokenize the prompt
        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(model.device)

        model.eval()

        # Generate the answer
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=500,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=(
                    tokenizer.pad_token_id
                    if tokenizer.pad_token_id is not None
                    else tokenizer.eos_token_id
                )
            )

        # Extract only newly generated tokens
        new_tokens = outputs[
            0, inputs["input_ids"].shape[1]:
        ]

        # Decode model response
        answer = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        # Remove complete thinking blocks, if any
        answer = re.sub(
            r"<think>.*?</think>",
            "",
            answer,
            flags=re.DOTALL
        ).strip()

        # Log diagnostic information in Colab
        print("\n" + "=" * 50)
        print("QUESTION:", question)
        print("GENERATED TOKENS:", len(new_tokens))
        print(
            "STOPPED AT EOS:",
            bool(
                len(new_tokens) > 0
                and new_tokens[-1].item() == tokenizer.eos_token_id
            )
        )
        print("MODEL ANSWER:", repr(answer))
        print("=" * 50)

        if not answer:
            raise HTTPException(
                status_code=500,
                detail="The model returned an empty answer."
            )

        return PromptResponse(answer=answer)

    except HTTPException:
        raise

    except Exception as exc:
        print("Generation error:", repr(exc))
        raise HTTPException(
            status_code=500,
            detail=f"Answer generation failed: {str(exc)}"
        )


print("OSTutorLLM prompt API is ready.")


OSTutorLLM prompt API is ready.


In [ ]:
print("Tokenizer:", "tokenizer" in globals())
print("Model:", "model" in globals())
print("API:", "app" in globals())

Tokenizer: True
Model: True
API: True


In [ ]:

# Stop any previous server on port 8000
!fuser -k 8000/tcp

print("Port cleanup complete.")


Port cleanup complete.


In [ ]:

from pyngrok import ngrok
from getpass import getpass
import threading
import time
import requests
import uvicorn

# Close old tunnels
ngrok.kill()

# Start FastAPI
server_thread = threading.Thread(
    target=uvicorn.run,
    kwargs={
        "app": app,
        "host": "0.0.0.0",
        "port": 8000,
        "log_level": "warning"
    },
    daemon=True
)
server_thread.start()

# Wait until the API responds
for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/",
            timeout=2
        )
        if response.status_code == 200:
            print("OSTutorLLM server is running.")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError(
        "API server did not start. Check the cell output."
    )

# Create the public tunnel
NGROK_TOKEN = getpass("Enter your ngrok authentication token: ")
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000, "http").public_url

print("OSTutorLLM API URL:", public_url)
print("API documentation:", public_url + "/docs")


OSTutorLLM server is running.
Enter your ngrok authentication token: ··········
OSTutorLLM API URL: https://crablike-cacti-baggy.ngrok-free.dev
API documentation: https://crablike-cacti-baggy.ngrok-free.dev/docs
